In [ ]:
import subprocess
import sys

import pandas as pd

from mpire import WorkerPool

SCRIPT_PATH = (
    "/share/home/zhoujb/project/riceDataComb/processData/"
    "1-0-4-Soybean_2795/0-0-0-1-run-plink2.sh"
)

def run_plink2snp(
    chr_name,
    start_bp,
    end_bp,
    region_name,
):
    cmd = [
        "bash",
        SCRIPT_PATH,
        str(chr_name),
        str(start_bp),
        str(end_bp),
        str(region_name),
    ]

    subprocess.run(
        cmd,
        check=True,
    )


if __name__ == "__main__":

    if len(sys.argv) < 3:
        print(
            "Usage: python 0-0-0-1-run-plink2.py "
            "<input_file_path> <num_workers>"
        )
        sys.exit(1)

    in_file_path = sys.argv[1]
    num_workers = int(sys.argv[2])

    # 读取区间文件
    info_file = pd.read_table(in_file_path)

    # 检查必要列
    required_cols = ["Chr", "Start", "End"]

    missing_cols = [
        col for col in required_cols
        if col not in info_file.columns
    ]

    if missing_cols:
        raise ValueError(
            f"Missing required columns: {missing_cols}"
        )

    input_list = []

    for index in info_file.index:

        chr_name = int(
            info_file.loc[index, "Chr"]
        )

        start_bp = int(
            info_file.loc[index, "Start"]
        )

        end_bp = int(
            info_file.loc[index, "End"]
        )

        # 基本区间检查
        if end_bp <= start_bp:
            print(
                f"Skip invalid region: "
                f"Chr{chr_name}:{start_bp}-{end_bp}"
            )
            continue

        # 每个区间生成唯一名称
        region_name = (
            f"Chr{chr_name}_{start_bp}_{end_bp}"
        )

        input_list.append(
            (
                chr_name,
                start_bp,
                end_bp,
                region_name,
            )
        )

    print(
        f"Total valid regions: {len(input_list)}"
    )

    # 并行运行
    with WorkerPool(
        n_jobs=num_workers
    ) as pool:

        pool.map(
            run_plink2snp,
            input_list,
            progress_bar=True,
        )

In [ ]:
#!/bin/bash

set -e

# Configure
ulimit -s unlimited

chr_name=$1
start_bp=$2
end_bp=$3
region_name=$4

source /share/home/zhoujb/anaconda3/etc/profile.d/conda.sh


GENOME_DIR="/share/home/zhoujb/project/riceDataComb/processData/1-0-4-Soybean_2795/rawData/rawGenome"

OUTPUT_DIR="/share/home/zhoujb/project/riceDataComb/processData/1-0-4-Soybean_2795/rawData/regionSNP"


# 创建输出目录
mkdir -p "${OUTPUT_DIR}"


# 当前染色体 PLINK 文件前缀
BFILE="${GENOME_DIR}/Chr${chr_name}.filterWild.biallele.snp"


echo "Processing:"
echo "  Chromosome: ${chr_name}"
echo "  Region: ${start_bp}-${end_bp}"
echo "  Input: ${BFILE}"
echo "  Output: ${region_name}"

set +e
plink2 \
    --bfile "${BFILE}" \
    --chr "${chr_name}" \
    --from-bp "${start_bp}" \
    --to-bp "${end_bp}" \
    --snps-only \
    --threads 1 \
    --export vcf \
    --out "${OUTPUT_DIR}/${region_name}"

exit_code=$?
set -e

if [ $exit_code -ne 0 ]; then
    echo "WARNING: plink2 exited with code ${exit_code} for ${region_name} (likely no variants in region). Skipping."
    exit 0
fi

echo "DONE: ${region_name}"
